In [11]:
#@title 0) Setup: folders, installs (lightweight)
import os, sys, json, shutil, zipfile, glob, re, random, math, textwrap
from pathlib import Path

# Folders
BASE = "/content"
DATA_DIR = os.path.join(BASE, "data")
OUT_DIR  = os.path.join(BASE, "out")
MODELS_DIR = os.path.join(OUT_DIR, "models")
for d in [DATA_DIR, OUT_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

# Minimal installs (keep it stable on Colab)
!pip -q install --upgrade nltk==3.9.1 sumy==0.11.0 langdetect==1.0.9 deep-translator==1.11.4

# NLTK resources
import nltk
nltk.download("punkt", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("universal_tagset", quiet=True)
nltk.download("vader_lexicon", quiet=True)

print("✅ Setup complete | Folders:", DATA_DIR, OUT_DIR, MODELS_DIR)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.3/97.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 97.2 MB/s eta 0:00:00
✅ Setup complete | Folders: /content/data /content/out /content/out/models


In [12]:
#@title 1) Load data: upload learn-ai-bbc.zip OR a CSV, then infer columns
from google.colab import files
import pandas as pd

# --- Option A: upload a ZIP that contains a CSV (e.g., learn-ai-bbc.zip) ---
print("👉 If you have a ZIP (e.g., learn-ai-bbc.zip) with a CSV inside, upload it now.")
up = files.upload()  # you can also skip and place files via the Files sidebar

# Extract any uploaded ZIPs into /content/data
for fname in up:
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(fname, "r") as z:
            z.extractall(DATA_DIR)
        print("✅ Extracted:", fname, "→", DATA_DIR)

# Find a CSV under /content/data or root
csv_candidates = []
for root in ["/content", DATA_DIR]:
    csv_candidates += glob.glob(os.path.join(root, "*.csv"))
    csv_candidates += glob.glob(os.path.join(root, "**/*.csv"), recursive=True)

if not csv_candidates:
    raise FileNotFoundError("No CSV found. Please upload a CSV or a ZIP containing a CSV.")

# Choose the 'best looking' BBC CSV (try names containing bbc/text/news)
def score_name(p):
    s = p.lower()
    score = 0
    for key in ["bbc", "text", "news"]:
        if key in s: score += 1
    return (-score, len(p))  # prefer more key hits, then shorter name

csv_path = sorted(csv_candidates, key=score_name)[0]
print("📄 Using CSV:", csv_path)

df_raw = pd.read_csv(csv_path)
print("Columns:", list(df_raw.columns))
df_raw.head()


👉 If you have a ZIP (e.g., learn-ai-bbc.zip) with a CSV inside, upload it now.


Saving learn-ai-bbc.zip to learn-ai-bbc (1).zip
✅ Extracted: learn-ai-bbc (1).zip → /content/data
📄 Using CSV: /content/data/BBC News Test.csv
Columns: ['ArticleId', 'Text']


,ArticleId,Text
0,1018,qpr keeper day heads for preston queens park r...
1,1319,software watching while you work software that...
2,1138,d arcy injury adds to ireland woe gordon d arc...
3,459,india s reliance family feud heats up the ongo...
4,1020,boro suffer morrison injury blow middlesbrough...


from matplotlib import pyplot as plt
_df_0['ArticleId'].plot(kind='hist', bins=20, title='ArticleId')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('Text').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['ArticleId'].plot(kind='line', figsize=(8, 4), title='ArticleId')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_3['Text'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_3, x='ArticleId', y='Text', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [24]:
OVERRIDE_TEXT_COL  = "text"      # or "content"
OVERRIDE_LABEL_COL = "category"  # or "label"


In [26]:
#@title 2) Normalize text + (auto-label if missing) → save bbc_clean.csv
import os, re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

# --------- CONFIG (only touch if needed) ----------
OVERRIDE_TEXT_COL  = None   # e.g., "Text"
OVERRIDE_LABEL_COL = None   # e.g., "Category"  (leave None if your CSV lacks labels)
MIN_PER_CLASS = 2           # ensure ≥2 items per class for stratified split
MAX_FEATURES  = 50000
# --------------------------------------------------

# Load the raw CSV chosen in Cell 1
csv_path = csv_path  # from Cell 1
df_raw = pd.read_csv(csv_path)

# 1) Detect or use overrides
def infer_columns(df):
    text_guess  = [c for c in df.columns if c.lower() in ["text","content","article","body","news"]]
    label_guess = [c for c in df.columns if c.lower() in ["category","label","topic","section","class"]]
    t = OVERRIDE_TEXT_COL or (text_guess[0] if text_guess else None)
    y = OVERRIDE_LABEL_COL or (label_guess[0] if label_guess else None)
    return t, y

text_col, label_col = infer_columns(df_raw)

# Special-case: your file has ['ArticleId','Text'] (no labels)
# If we still don't have a label_col, we will create one via clustering.
if text_col is None:
    # try exact 'Text' fallback (from your screenshot)
    if "Text" in df_raw.columns:
        text_col = "Text"

if text_col is None:
    print("Columns found:", list(df_raw.columns))
    raise ValueError("No text column found. Set OVERRIDE_TEXT_COL to the text field name and re-run.")

# Build base df
df = df_raw[[text_col]].copy().rename(columns={text_col: "text"})
df["text"] = df["text"].astype(str).str.strip()
df = df.dropna(subset=["text"])
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# 2) If we already have labels, use them; otherwise auto-label via clustering
if label_col and (label_col in df_raw.columns):
    y = df_raw[label_col].astype(str).str.strip()
    df["category"] = y.loc[df.index].fillna("misc").replace("", "misc")
else:
    # ---- Auto-label: TF-IDF + MiniBatchKMeans ----
    # pick a reasonable cluster count k
    n = len(df)
    # heuristic: 5 clusters, but adapt for small datasets
    k = min(5, max(2, n // 250 if n >= 600 else (3 if n >= 120 else 2)))

    tfidf = TfidfVectorizer(stop_words="english", max_features=MAX_FEATURES, min_df=2, max_df=0.95)
    X = tfidf.fit_transform(df["text"])

    km = MiniBatchKMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X)

    # Build readable names from top terms
    terms = np.array(tfidf.get_feature_names_out())
    centers = km.cluster_centers_
    names = []
    topk = 6
    for i in range(k):
        idx = centers[i].argsort()[::-1][:topk]
        top_words = [t for t in terms[idx] if re.match(r"^[a-z]{3,}$", t)]
        names.append(f"topic_{i}:" + ",".join(top_words[:4]) if top_words else f"topic_{i}")

    # map numeric label → name
    cluster_name = {i: names[i] for i in range(k)}
    df["category"] = pd.Series(labels, index=df.index).map(cluster_name)

# 3) Ensure each class has ≥ MIN_PER_CLASS rows (merge tiny ones to 'misc')
vc = df["category"].value_counts()
tiny = vc[vc < MIN_PER_CLASS].index.tolist()
if tiny:
    df.loc[df["category"].isin(tiny), "category"] = "misc"

# If merging produced only one class, force a second class by splitting misc (rare edge case)
if df["category"].nunique() == 1 and len(df) >= 4:
    half = len(df) // 2
    df.loc[df.index[:half], "category"] = df.loc[df.index[:half], "category"].astype(str) + "_A"
    df.loc[df.index[half:], "category"] = df.loc[df.index[half:], "category"].astype(str) + "_B"

print("✅ Final classes & counts:\n", df["category"].value_counts())

# 4) Save clean CSV
clean_path = os.path.join(DATA_DIR, "bbc_clean.csv")
df.to_csv(clean_path, index=False)
print(f"✅ Saved clean CSV → {clean_path}")
df.head()


✅ Final classes & counts:
 category
topic_0:said,government,people,labour    408
topic_1:said,film,game,best              314
Name: count, dtype: int64
✅ Saved clean CSV → /content/data/bbc_clean.csv


,text,category
0,qpr keeper day heads for preston queens park r...,"topic_1:said,film,game,best"
1,software watching while you work software that...,"topic_0:said,government,people,labour"
2,d arcy injury adds to ireland woe gordon d arc...,"topic_1:said,film,game,best"
3,india s reliance family feud heats up the ongo...,"topic_0:said,government,people,labour"
4,boro suffer morrison injury blow middlesbrough...,"topic_1:said,film,game,best"


In [27]:
#@title 3) Train/test split (stratified)
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(os.path.join(DATA_DIR,"bbc_clean.csv"))
vc = df["category"].value_counts()
if (vc < 2).any():
    raise ValueError("Each class needs ≥2 examples for stratified split. Found:\n" + str(vc))

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["category"], test_size=0.2, random_state=42, stratify=df["category"]
)
print("📊 Train size:", len(X_train), "| Test size:", len(X_test))
print("Train class counts:\n", pd.Series(y_train).value_counts())


📊 Train size: 577 | Test size: 145
Train class counts:
 category
topic_0:said,government,people,labour    326
topic_1:said,film,game,best              251
Name: count, dtype: int64


In [28]:
#@title 4) Train models, evaluate, and save best pipeline + metrics.json
import pandas as pd, json, joblib, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

pipelines = {
    "LogReg": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_df=0.9, max_features=50000)),
        ("clf", LogisticRegression(max_iter=200, n_jobs=None))
    ]),
    "LinearSVC": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_df=0.9, max_features=50000)),
        ("clf", LinearSVC())
    ])
}

reports, accuracies = {}, {}
best_name, best_acc = None, -1.0

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies[name] = float(acc)
    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    reports[name] = rep
    print(f"🔹 {name} accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc, best_name = acc, name
        best_pipe = pipe

# Save best pipeline + metrics
joblib.dump(best_pipe, os.path.join(MODELS_DIR, f"{best_name}_pipeline.joblib"))
metrics = {
    "best_model": best_name,
    "accuracy": accuracies,
    "n_train": int(len(X_train)),
    "n_test":  int(len(X_test)),
    "labels": sorted(df["category"].unique().tolist()),
    "report_best": reports[best_name]
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("🏁 Best model:", best_name, "| Saved →", MODELS_DIR)
print("📝 metrics.json →", os.path.join(OUT_DIR, "metrics.json"))


🔹 LogReg accuracy: 0.9586
🔹 LinearSVC accuracy: 0.9724
🏁 Best model: LinearSVC | Saved → /content/out/models
📝 metrics.json → /content/out/metrics.json


In [29]:
#@title 5) Topic Modeling (NMF) → topics_top_words.csv + doc_topic_distributions.csv
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

df = pd.read_csv(os.path.join(DATA_DIR,"bbc_clean.csv"))

tfidf = TfidfVectorizer(stop_words="english", max_features=50000, min_df=2, max_df=0.9)
X = tfidf.fit_transform(df["text"])

N_TOPICS = 10  # adjust if needed
nmf = NMF(n_components=N_TOPICS, init="nndsvda", random_state=42, max_iter=300)
W = nmf.fit_transform(X)   # doc-topic
H = nmf.components_        # topic-term

terms = np.array(tfidf.get_feature_names_out())
topk = 12
rows = []
for t in range(N_TOPICS):
    idx = H[t].argsort()[::-1][:topk]
    rows.append({"topic": t, "top_words": ", ".join(terms[idx])})

topics_df = pd.DataFrame(rows)
doc_topics = pd.DataFrame(W, columns=[f"topic_{i}" for i in range(N_TOPICS)])
doc_topics.insert(0, "category", df["category"])
doc_topics.insert(1, "text_snippet", df["text"].str.slice(0,140))

topics_df.to_csv(os.path.join(OUT_DIR,"topics_top_words.csv"), index=False)
doc_topics.to_csv(os.path.join(OUT_DIR,"doc_topic_distributions.csv"), index=False)
print("✅ Saved topics_top_words.csv and doc_topic_distributions.csv to", OUT_DIR)
topics_df.head()


✅ Saved topics_top_words.csv and doc_topic_distributions.csv to /content/out


,topic,top_words
0,0,"race, indoor, win, champion, title, world, fin..."
1,1,"mr, labour, blair, election, party, brown, how..."
2,2,"economy, growth, economic, prices, said, year,..."
3,3,"mobile, technology, people, digital, broadband..."
4,4,"film, festival, award, films, oscar, actress, ..."


In [30]:
#@title 6) POS analysis with NLTK (sampled per class) → pos_by_category.csv
import pandas as pd, numpy as np, random
import nltk
from nltk import word_tokenize, pos_tag

# safety: ensure resources exist
for pkg in ["punkt","averaged_perceptron_tagger","universal_tagset"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}") if pkg=="punkt" else nltk.data.find(f"taggers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)

df = pd.read_csv(os.path.join(DATA_DIR,"bbc_clean.csv"))
SAMPLE_PER_CLASS = 60  # tweak for speed/depth

records = []
for cat, sub in df.groupby("category"):
    sample = sub.sample(min(SAMPLE_PER_CLASS, len(sub)), random_state=42)
    pos_counts = {}
    for text in sample["text"].tolist():
        tokens = word_tokenize(text)
        tags = pos_tag(tokens, tagset="universal")
        for _, t in tags:
            pos_counts[t] = pos_counts.get(t, 0) + 1
    total = sum(pos_counts.values()) or 1
    for tag, cnt in sorted(pos_counts.items()):
        records.append({"category": cat, "pos": tag, "count": cnt, "pct": cnt/total})

pos_df = pd.DataFrame(records).sort_values(["category","count"], ascending=[True, False])
pos_df.to_csv(os.path.join(OUT_DIR,"pos_by_category.csv"), index=False)
print("✅ Saved →", os.path.join(OUT_DIR,"pos_by_category.csv"))
pos_df.head()


✅ Saved → /content/out/pos_by_category.csv


,category,pos,count,pct
6,"topic_0:said,government,people,labour",NOUN,8523,0.271597
10,"topic_0:said,government,people,labour",VERB,6183,0.197030
2,"topic_0:said,government,people,labour",ADP,3658,0.116567
5,"topic_0:said,government,people,labour",DET,3093,0.098563
1,"topic_0:said,government,people,labour",ADJ,2959,0.094293


In [31]:
#@title 7) Summarization with LexRank (sumy) → summaries_sample.csv
import pandas as pd, random, textwrap
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

df = pd.read_csv(os.path.join(DATA_DIR,"bbc_clean.csv"))
summarizer = LexRankSummarizer()
SAMPLES = 40
SENTENCES = 3

rows = []
for _, row in df.sample(min(SAMPLES, len(df)), random_state=42).iterrows():
    text = row["text"]
    try:
        parser = PlaintextParser.from_string(text, Tokenizer("english"))
        sents = summarizer(parser.document, SENTENCES)
        summary = " ".join(str(s) for s in sents)
    except Exception:
        summary = textwrap.shorten(text, width=350, placeholder="…")
    rows.append({"category": row["category"], "summary": summary, "orig_len": len(text)})

summ_df = pd.DataFrame(rows)
summ_df.to_csv(os.path.join(OUT_DIR,"summaries_sample.csv"), index=False)
print("✅ Saved →", os.path.join(OUT_DIR,"summaries_sample.csv"))
summ_df.head()


✅ Saved → /content/out/summaries_sample.csv


,category,summary,orig_len
0,"topic_1:said,film,game,best",running around the olympics it was back to off...,3101
1,"topic_0:said,government,people,labour",the global sports giant said it posted a profi...,1367
2,"topic_0:said,government,people,labour",mr balls rejected the allegation that mr brown...,1736
3,"topic_0:said,government,people,labour",us bank in $515m sec settlement five bank of a...,2381
4,"topic_0:said,government,people,labour",oasis star fined for german brawl oasis singer...,1132


In [32]:
#@title 8) Semantic search (TF-IDF cosine) → search_examples.csv
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv(os.path.join(DATA_DIR,"bbc_clean.csv"))

vec = TfidfVectorizer(stop_words="english", max_features=50000, min_df=2, max_df=0.9)
X = vec.fit_transform(df["text"])

def search(query, k=5):
    qv = vec.transform([query])
    sims = cosine_similarity(qv, X)[0]
    idx = np.argsort(-sims)[:k]
    return df.iloc[idx][["category","text"]].assign(score=sims[idx])

queries = [
    "premier league match report",
    "new smartphone release and software update",
    "government budget and economic policy",
]
rows = []
for q in queries:
    hits = search(q, k=5)
    for _, r in hits.iterrows():
        rows.append({"query": q, "category": r["category"], "score": float(r["score"]), "snippet": r["text"][:180]})


In [33]:
#@title 9) Multilingual: detect → translate → sentiment → save multilingual_eval.csv
import pandas as pd, numpy as np
from langdetect import detect
from deep_translator import GoogleTranslator
from nltk.sentiment import SentimentIntensityAnalyzer

df = pd.read_csv(os.path.join(DATA_DIR,"bbc_clean.csv"))

# small demo set; you can add your own non-English texts here
samples = [
    "El gobierno anuncia nuevas medidas económicas para apoyar a las pequeñas empresas.",
    "La selección ganó el partido con un gol en el último minuto.",
    "Le smartphone présente une mise à jour logicielle majeure améliorant la sécurité.",
]
# seed with 3 English too
samples += df["text"].sample(min(3, len(df)), random_state=42).tolist()

sia = SentimentIntensityAnalyzer()
rows = []
for txt in samples:
    try:
        lang = detect(txt)
    except:
        lang = "unknown"
    if lang != "en":
        try:
            eng = GoogleTranslator(source="auto", target="en").translate(txt)
        except Exception:
            eng = txt  # fallback
    else:
        eng = txt
    sent = sia.polarity_scores(eng)["compound"]
    rows.append({"orig_lang": lang, "original": txt[:200], "english": eng[:200], "sentiment_compound": sent})

ml_df = pd.DataFrame(rows)
ml_df.to_csv(os.path.join(OUT_DIR,"multilingual_eval.csv"), index=False)
print("✅ Saved →", os.path.join(OUT_DIR,"multilingual_eval.csv"))
ml_df.head()


✅ Saved → /content/out/multilingual_eval.csv


,orig_lang,original,english,sentiment_compound
0,es,El gobierno anuncia nuevas medidas económicas ...,The government announces new economic measures...,0.4019
1,es,La selección ganó el partido con un gol en el ...,The team won the match with a goal in the last...,0.5719
2,fr,Le smartphone présente une mise à jour logicie...,The smartphone features a major software updat...,0.6369
3,en,running around the olympics it was back to off...,running around the olympics it was back to off...,0.9926
4,en,strong quarterly growth for nike nike has repo...,strong quarterly growth for nike nike has repo...,0.9910


In [34]:
#@title 10) Inference demo: load best pipeline and classify examples → predictions.csv
import json, joblib, pandas as pd, os, glob

with open(os.path.join(OUT_DIR,"metrics.json")) as f:
    metrics = json.load(f)
best_name = metrics["best_model"]
pipe_path = os.path.join(MODELS_DIR, f"{best_name}_pipeline.joblib")
pipe = joblib.load(pipe_path)

examples = [
    "The central bank raised interest rates to combat inflation concerns.",
    "The team secured a thrilling victory in the championship final.",
    "A new software update brings performance improvements to the device.",
]
preds = pipe.predict(examples)
pd.DataFrame({"text": examples, "predicted_category": preds}).to_csv(os.path.join(OUT_DIR,"predictions.csv"), index=False)
print("✅ Saved →", os.path.join(OUT_DIR,"predictions.csv"))


✅ Saved → /content/out/predictions.csv


In [35]:
#@title 🔧 Setup helpers (robust column detection & paths)
import os, pandas as pd, numpy as np, json, joblib, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns

DATA_DIR = "/content/data"
OUT_DIR  = "/content/out"
MODELS_DIR = os.path.join(OUT_DIR, "models")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def load_clean_df():
    csv_path = os.path.join(DATA_DIR, "bbc_clean.csv")
    df = pd.read_csv(csv_path)
    # detect text & label columns in a tolerant way
    cols = {c.lower(): c for c in df.columns}
    text_col  = cols.get("text") or cols.get("content") or cols.get("headline") or cols.get("body")
    label_col = cols.get("category") or cols.get("label") or cols.get("topic")
    if not text_col:
        raise ValueError(f"Could not find a text column in {list(df.columns)}")
    if not label_col:
        # if no label, create a dummy single class to keep pipelines running
        label_col = "__auto_label__"
        df[label_col] = "unlabeled"
    return df, text_col, label_col

def quick_train_or_load(df, text_col, label_col):
    """Load tfidf+logreg if exists, otherwise fit quickly and save."""
    tfidf_p = os.path.join(MODELS_DIR, "tfidf.joblib")
    best_p  = os.path.join(MODELS_DIR, "best_logreg.joblib")
    metr_p  = os.path.join(OUT_DIR, "metrics.json")

    if os.path.exists(tfidf_p) and os.path.exists(best_p):
        tfidf = joblib.load(tfidf_p)
        clf   = joblib.load(best_p)
        return tfidf, clf

    X_train, X_test, y_train, y_test = train_test_split(
        df[text_col], df[label_col], test_size=0.2, random_state=42, stratify=df[label_col] if df[label_col].nunique()>1 else None
    )
    tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=2, max_df=0.9, max_features=50000)
    Xtr   = tfidf.fit_transform(X_train)
    clf   = LogisticRegression(max_iter=1000, n_jobs= -1 if hasattr(LogisticRegression(), "n_jobs") else None)
    clf.fit(Xtr, y_train)

    # eval
    y_pred = clf.predict(tfidf.transform(X_test))
    acc = float(accuracy_score(y_test, y_pred))
    rep = classification_report(y_test, y_pred, output_dict=True)

    # save artifacts
    joblib.dump(tfidf, tfidf_p)
    joblib.dump(clf,   best_p)
    with open(metr_p, "w") as f:
        json.dump({"best_model":"best_logreg.joblib","accuracy":acc,"report":rep}, f, indent=2)

    # confusion matrix PNG
    cm = confusion_matrix(y_test, y_pred, labels=sorted(df[label_col].unique()))
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=sorted(df[label_col].unique()), yticklabels=sorted(df[label_col].unique()))
    plt.title("Confusion Matrix (LogReg + TF-IDF)")
    plt.ylabel("True"); plt.xlabel("Predicted")
    cm_path = os.path.join(OUT_DIR, "confusion_matrix.png")
    plt.tight_layout(); plt.savefig(cm_path); plt.close()

    # misclass table
    mis = pd.DataFrame({"text": X_test.reset_index(drop=True), "true": y_test.reset_index(drop=True), "pred": y_pred})
    mis = mis[mis["true"]!=mis["pred"]]
    mis.to_csv(os.path.join(OUT_DIR,"misclassified_samples.csv"), index=False)

    print(f"✅ Trained & saved → {tfidf_p}, {best_p}, metrics.json, confusion_matrix.png, misclassified_samples.csv")
    return tfidf, clf


In [36]:
#@title 🧩 Topic Modeling (NMF + LDA) — saves topics & per-doc weights
df, text_col, label_col = load_clean_df()

# TF-IDF for NMF topics (best with tf-idf)
tfidf = TfidfVectorizer(stop_words="english", max_df=0.95, min_df=2, max_features=30000)
X_tfidf = tfidf.fit_transform(df[text_col].astype(str))

n_topics = 10  # adjust if you want
nmf = NMF(n_components=n_topics, random_state=42, init="nndsvda", max_iter=400)
W = nmf.fit_transform(X_tfidf)
H = nmf.components_

# Top terms per topic
terms = np.array(tfidf.get_feature_names_out())
topn = 12
nmf_topics = []
for k, comp in enumerate(H):
    top_terms = terms[np.argsort(comp)[::-1][:topn]]
    nmf_topics.append({"model":"NMF", "topic":k, "top_terms":", ".join(top_terms)})

# LDA on bag-of-words (CountVectorizer)
cv = CountVectorizer(stop_words="english", max_df=0.95, min_df=2, max_features=30000)
X_bow = cv.fit_transform(df[text_col].astype(str))
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method="batch", max_iter=20)
lda_W = lda.fit_transform(X_bow)
lda_H = lda.components_
vocab = np.array(cv.get_feature_names_out())
lda_topics = []
for k, comp in enumerate(lda_H):
    top_terms = vocab[np.argsort(comp)[::-1][:topn]]
    lda_topics.append({"model":"LDA", "topic":k, "top_terms":", ".join(top_terms)})

topics_df = pd.DataFrame(nmf_topics + lda_topics)
topics_df.to_csv(os.path.join(OUT_DIR,"topics_overview.csv"), index=False)

# Per-document topic weights (NMF) to inspect dominant themes
doc_topics = pd.DataFrame(W, columns=[f"nmf_topic_{i}" for i in range(n_topics)])
doc_topics.insert(0, "doc_id", np.arange(len(df)))
doc_topics.to_csv(os.path.join(OUT_DIR,"doc_topics_nmf.csv"), index=False)

print("✅ Saved:", "topics_overview.csv,", "doc_topics_nmf.csv")
topics_df.head()


✅ Saved: topics_overview.csv, doc_topics_nmf.csv


,model,topic,top_terms
0,NMF,0,"race, indoor, win, champion, title, world, fin..."
1,NMF,1,"mr, labour, blair, election, party, brown, how..."
2,NMF,2,"economy, growth, economic, prices, said, year,..."
3,NMF,3,"mobile, technology, people, digital, broadband..."
4,NMF,4,"film, festival, award, films, oscar, actress, ..."


In [37]:
#@title 📝 Summarization (TextRank: sumy) — saves summaries.csv
!pip -q install sumy

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer

df, text_col, label_col = load_clean_df()
summarizer = TextRankSummarizer()
n_sentences = 3  # per summary
limit = min(200, len(df))  # keep it fast; adjust higher if you want

rows = []
for i, row in df.head(limit).iterrows():
    text = str(row[text_col])
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summary = " ".join(str(s) for s in summarizer(parser.document, n_sentences))
    rows.append({"doc_id": i, "summary": summary, "label": row.get(label_col, "")})

summ_df = pd.DataFrame(rows)
summ_df.to_csv(os.path.join(OUT_DIR, "summaries.csv"), index=False)
print("✅ Saved:", os.path.join(OUT_DIR, "summaries.csv"))
summ_df.head()


✅ Saved: /content/out/summaries.csv


,doc_id,summary,label
0,0,qpr keeper day heads for preston queens park r...,"topic_1:said,film,game,best"
1,1,sabotage and data theft are most commonly comm...,"topic_0:said,government,people,labour"
2,2,girvan dempsey comes into the team to take the...,"topic_1:said,film,game,best"
3,3,india s reliance family feud heats up the ongo...,"topic_0:said,government,people,labour"
4,4,morrison was sent for scans after being substi...,"topic_1:said,film,game,best"


In [38]:
#@title 🔎 Semantic Search (TF-IDF + cosine) — saves search_examples.csv
df, text_col, label_col = load_clean_df()

tfidf = TfidfVectorizer(stop_words="english", max_features=50000)
X = tfidf.fit_transform(df[text_col].astype(str))

def semantic_search(query, k=5):
    q = tfidf.transform([query])
    sims = cosine_similarity(q, X).ravel()
    idx = np.argsort(-sims)[:k]
    return pd.DataFrame({
        "rank": np.arange(1, k+1),
        "score": sims[idx],
        "doc_id": idx,
        "text": df.iloc[idx][text_col].values,
        "label": df.iloc[idx][label_col].values if label_col in df.columns else ""
    })

# save a few example queries
examples = ["election results", "space mission", "financial markets", "football match"]
all_rows = []
for q in examples:
    res = semantic_search(q, k=5)
    res.insert(0, "query", q)
    all_rows.append(res)
out_search = pd.concat(all_rows, ignore_index=True)
out_search.to_csv(os.path.join(OUT_DIR, "search_examples.csv"), index=False)

print("✅ Saved:", os.path.join(OUT_DIR, "search_examples.csv"))
semantic_search("technology startup funding", k=5).head(3)


✅ Saved: /content/out/search_examples.csv


,rank,score,doc_id,text,label
0,1,0.141550,387,why cell will get the hard sell the world is c...,"topic_1:said,film,game,best"
1,2,0.118818,582,technology gets the creative bug the hi-tech a...,"topic_0:said,government,people,labour"
2,3,0.110925,656,a question of trust and technology a major gov...,"topic_0:said,government,people,labour"


In [39]:
#@title 🌍 Multilingual sample (translate → sentiment) — saves multilingual_eval.csv
!pip -q install deep-translator
import nltk, math
from deep_translator import GoogleTranslator
from nltk.sentiment import SentimentIntensityAnalyzer

# ensure resources
nltk.download("vader_lexicon", quiet=True)
nltk.download("punkt", quiet=True)

df, text_col, label_col = load_clean_df()
sample_texts = [
    "La economía global muestra signos de recuperación.",
    "El equipo perdió el partido en los últimos minutos.",
    "La tecnología de inteligencia artificial avanza rápidamente.",
    "La película fue aburrida y demasiado larga."
]
# translate to English
translated = [GoogleTranslator(source="auto", target="en").translate(t) for t in sample_texts]

sia = SentimentIntensityAnalyzer()
rows = []
for src, eng in zip(sample_texts, translated):
    sc = sia.polarity_scores(eng)
    rows.append({"source_text": src, "translated_en": eng, **sc})

ml_df = pd.DataFrame(rows)
ml_df.to_csv(os.path.join(OUT_DIR, "multilingual_eval.csv"), index=False)
print("✅ Saved:", os.path.join(OUT_DIR, "multilingual_eval.csv"))
ml_df


✅ Saved: /content/out/multilingual_eval.csv


,source_text,translated_en,neg,neu,pos,compound
0,La economía global muestra signos de recuperac...,The global economy shows signs of recovery.,0.000,1.000,0.000,0.0000
1,El equipo perdió el partido en los últimos min...,The team lost the game in the last minutes.,0.223,0.777,0.000,-0.3182
2,La tecnología de inteligencia artificial avanz...,Artificial intelligence technology is advancin...,0.000,0.617,0.383,0.4767
3,La película fue aburrida y demasiado larga.,The movie was boring and too long.,0.277,0.723,0.000,-0.3182
